In [4]:
class PuzzleState:
    def __init__(self, board, parent=None, move="", depth=0):
        self.board = board       
        self.parent = parent    
        self.move = move        
        self.depth = depth      

    def __eq__(self, other):
        return self.board == other.board

    def __hash__(self):
        
        return hash(self.board)

def get_neighbors(state):
    neighbors = []
    board = list(state.board)
    blank_idx = board.index(0)
    row, col = divmod(blank_idx, 3)

    moves = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]

    for dr, dc, move_name in moves:
        new_row, new_col = row + dr, col + dc
        
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_blank_idx = new_row * 3 + new_col
            new_board = board[:]
            new_board[blank_idx], new_board[new_blank_idx] = new_board[new_blank_idx], new_board[blank_idx]
            
            neighbors.append(PuzzleState(tuple(new_board), state, move_name, state.depth + 1))
            
    return neighbors

def solve_dfs(start_board, goal_board, max_depth=20):
    start_state = PuzzleState(tuple(start_board))
    goal_state = PuzzleState(tuple(goal_board))

    stack = [start_state]
    visited = {start_state.board: 0}

    while stack:
        current_state = stack.pop()

        if current_state.board == goal_state.board:
            return reconstruct_path(current_state)

        if current_state.depth >= max_depth:
            continue

        for neighbor in get_neighbors(current_state):
            if neighbor.board not in visited or neighbor.depth < visited[neighbor.board]:
                visited[neighbor.board] = neighbor.depth
                stack.append(neighbor)

    return None  

def reconstruct_path(state):
    path = []
    current = state
    while current.parent is not None:
        path.append((current.move, current.board))
        current = current.parent
    path.reverse()
    return path

def print_board(board):
    for i in range(0, 9, 3):
        print(f"{board[i]} {board[i+1]} {board[i+2]}")
    print("-" * 10)

if __name__ == "__main__":
    initial_state = [
        1, 2, 3,
        4, 0, 6,
        7, 5, 8
    ]

    target_goal = [
        1, 2, 3,
        4, 5, 6,
        7, 8, 0
    ]

    print("Initial Board Position:")
    print_board(initial_state)

    solution_path = solve_dfs(initial_state, target_goal, max_depth=15)

    if solution_path:
        print(f"Goal Reached in {len(solution_path)} steps!\n")
        for step, (move, board) in enumerate(solution_path, 1):
            print(f"Step {step}: Move Blank '{move}'")
            print_board(board)
    else:
        print("No solution found within the specified depth limit.")


Initial Board Position:
1 2 3
4 0 6
7 5 8
----------
Goal Reached in 10 steps!

Step 1: Move Blank 'Right'
1 2 3
4 6 0
7 5 8
----------
Step 2: Move Blank 'Down'
1 2 3
4 6 8
7 5 0
----------
Step 3: Move Blank 'Left'
1 2 3
4 6 8
7 0 5
----------
Step 4: Move Blank 'Up'
1 2 3
4 0 8
7 6 5
----------
Step 5: Move Blank 'Right'
1 2 3
4 8 0
7 6 5
----------
Step 6: Move Blank 'Down'
1 2 3
4 8 5
7 6 0
----------
Step 7: Move Blank 'Left'
1 2 3
4 8 5
7 0 6
----------
Step 8: Move Blank 'Up'
1 2 3
4 0 5
7 8 6
----------
Step 9: Move Blank 'Right'
1 2 3
4 5 0
7 8 6
----------
Step 10: Move Blank 'Down'
1 2 3
4 5 6
7 8 0
----------


In [5]:
import itertools

GOAL_STATE = (1, 2, 3, 4, 5, 6, 7, 8, 0)

def get_moves(state):
    """Generates all valid next states from the current state."""
    moves = []
    zero_index = state.index(0)
    row, col = divmod(zero_index, 3)
    
    directions = {
        "UP": (-1, 0),
        "DOWN": (1, 0),
        "LEFT": (0, -1),
        "RIGHT": (0, 1)
    }
    
    for direction, (dr, dc) in directions.items():
        new_row, new_col = row + dr, col + dc
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            swap_index = new_row * 3 + new_col
            
            new_state = list(state)
            new_state[zero_index], new_state[swap_index] = new_state[swap_index], new_state[zero_index]
            moves.append(tuple(new_state))
            
    return moves

def depth_limited_dfs(state, goal, depth, path):
    """Performs a standard DFS constrained by a maximum depth limit."""
    if state == goal:
        return path
    if depth <= 0:
        return None
        
    for neighbor in get_moves(state):
        if neighbor not in path:
            result = depth_limited_dfs(neighbor, goal, depth - 1, path + (neighbor,))
            if result is not None:
                return result
    return None

def iddfs(start_state, goal_state):
    """Iteratively deepens the DFS depth limit until a solution is found."""
    for depth in itertools.count():
        result = depth_limited_dfs(start_state, goal_state, depth, (start_state,))
        if result is not None:
            return result, depth

def print_board(state):
    """Helper function to print a flat state tuple in a 3x3 grid."""
    for i in range(0, 9, 3):
        print(f" {state[i]} {state[i+1]} {state[i+2]} ")
    print("-" * 11)

if __name__ == "__main__":
    initial_state = (1, 2, 3, 4, 0, 6, 7, 5, 8)
    
    print("Initial State:")
    print_board(initial_state)
    
    print("Searching for optimal solution using IDDFS...")
    solution_path, total_moves = iddfs(initial_state, GOAL_STATE)
    
    print(f"\nSuccess! Solution found in {total_moves} moves.\n")
    for step, state in enumerate(solution_path):
        print(f"Step {step}:")
        print_board(state)


Initial State:
 1 2 3 
 4 0 6 
 7 5 8 
-----------
Searching for optimal solution using IDDFS...

Success! Solution found in 2 moves.

Step 0:
 1 2 3 
 4 0 6 
 7 5 8 
-----------
Step 1:
 1 2 3 
 4 5 6 
 7 0 8 
-----------
Step 2:
 1 2 3 
 4 5 6 
 7 8 0 
-----------
